# RAG Retrieval Demo
## End-to-end pipeline walkthrough in mock mode

This notebook demonstrates the full RAG retrieval pipeline **step by step**, from a raw
text file to ranked retrieved chunks. Everything runs in **mock mode** — no API keys,
no model downloads, no internet connection required.

---

### What you will see

| Step | Component | Output |
|------|-----------|--------|
| 1 | Document Loader | `Document` dataclass |
| 2 | Text Cleaner | normalised string |
| 3 | RecursiveChunker | `List[Chunk]` |
| 4 | MockEmbeddingProvider | 384-dim unit vectors |
| 5 | ChromaVectorStore | chunks stored with metadata |
| 6 | RAGPipeline.query() | top-k results |
| 7 | Inspection | rank, score, doc_id, preview |

---

### Prerequisites

Run from the **repository root** (not from inside `notebooks/`):
```
jupyter notebook notebooks/rag_retrieval_demo.ipynb
```
or open in VS Code with the Jupyter extension.

## Pipeline Architecture

The pipeline has five sequential layers, each with a well-defined interface.
`RAGPipeline` is the thin orchestrator — it owns no logic of its own:

```
Raw document (.txt / .pdf / .html)
        |
        v
  DocumentLoader        ->  Document(doc_id, text, source, file_type)
        |
        v
  TextCleaner           ->  normalised, whitespace-cleaned text
        |
        v
  Chunker               ->  List[Chunk(chunk_id, doc_id, text)]
        |
        v
  EmbeddingProvider     ->  chunk.embedding = List[float]  (dim=384)
        |
        v
  VectorStore.add()     ->  vectors + metadata persisted
        |
  query text in
        |
        v
  Retriever.query()     ->  embed -> search -> (optional rerank)
        |
        v
  List[Chunk]  with score and rank populated
```

The same `EmbeddingProvider` instance is used for both `build_index()` and `query()`,
guaranteeing that query embeddings live in the same vector space as index embeddings.

## Why Mock Mode Exists

Real embedding providers have two drawbacks for development and evaluation:

| Provider | Drawback |
|----------|----------|
| OpenAI / Gemini | Require API keys; incur cost per call |
| HuggingFace sentence-transformers | Download ~90 MB+ model on first use |

`MockEmbeddingProvider` solves both:
- Uses a **deterministic SHA-256-seeded NumPy RNG** to generate L2-normalised 384-dim vectors
- Requires **no API key, no internet, no model download**
- Produces the **same vector for the same text** — stable across runs
- Makes tests and notebooks **reproducible and fast**

> **Note:** Because mock embeddings are seeded by text hash (not semantic meaning),
> retrieval rankings are not semantically meaningful. Use HuggingFace or OpenAI
> providers for real semantic retrieval experiments.

In [ ]:
# ---------------------------------------------------------------------------
# Environment setup — run this cell first
# ---------------------------------------------------------------------------
import sys
import os

# Support running from repo root or from notebooks/ subdirectory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(os.path.abspath('..'))

# Add repo root to sys.path so src.* imports resolve
repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print(f'Working directory : {os.getcwd()}')
print(f'Python            : {sys.version.split()[0]}')

## Step 1 — Load a Sample Document

`load_document()` reads a file from disk and returns a `Document` dataclass.
The `doc_id` is derived from the **file stem** (filename without extension), so
a file named `sample_doc.txt` produces `doc_id = 'sample_doc'`.

Supported file types: `.txt`, `.pdf` (via pypdf), `.html` / `.htm` (via BeautifulSoup).

In [1]:
import tempfile
from src.ingestion.document_loader import load_document

# Sample document covering all ground-truth question topics
SAMPLE_TEXT = (
    'This system supports two chunking strategies: recursive and fixed-size.\n'
    'Recursive chunking splits text by natural boundaries (paragraphs, sentences)\n'
    'before falling back to word and character splits. Fixed chunking splits text\n'
    'by a fixed character count with configurable overlap.\n\n'
    'The recommended chunk overlap baseline is 10 to 20 percent of the chunk size.\n'
    'For example, with chunk_size=500, an overlap of 50 to 100 characters is typical.\n\n'
    'Four embedding providers are available: mock, HuggingFace, OpenAI, and Gemini.\n'
    'Mock mode uses a deterministic SHA-256-seeded vector and requires no API key.\n'
    'It is the safe default for local and offline execution.\n\n'
    'The system supports two vector stores: ChromaDB and FAISS.\n'
    'ChromaDB persists to disk automatically. FAISS requires an explicit persist call.\n'
    'Both expose the same VectorStore interface, so they are interchangeable.\n\n'
    'Retrieval quality is measured with four standard IR metrics:\n'
    'Recall@K, Precision@K, MRR (Mean Reciprocal Rank), and nDCG@K.\n'
)

# Write to a named temp file so load_document() can derive doc_id from the file stem
_tmpdir = tempfile.mkdtemp(prefix='rag_demo_')
_doc_path = os.path.join(_tmpdir, 'sample_doc.txt')

with open(_doc_path, 'w', encoding='utf-8') as fh:
    fh.write(SAMPLE_TEXT)

document = load_document(_doc_path)

print(f'doc_id    : {document.doc_id}')
print(f'file_type : {document.file_type}')
print(f'source    : {document.source}')
print(f'text len  : {len(document.text)} chars')
print(f'\nFirst 200 chars:')
print('-' * 40)
print(document.text[:200])

ModuleNotFoundError: No module named 'src'

## Step 2 — Clean Text

`clean_text()` performs lightweight normalisation:
- Collapses runs of whitespace
- Strips leading / trailing whitespace
- Optionally redacts PII patterns (disabled by default)

The cleaned text is written back to `document.text` before chunking.

In [ ]:
from src.ingestion.text_cleaner import clean_text

original_len = len(document.text)
cleaned = clean_text(document.text, redact_pii=False)

# Write cleaned text back to the document
document.text = cleaned

print(f'Length before clean : {original_len} chars')
print(f'Length after clean  : {len(cleaned)} chars')
print(f'\nCleaned text preview (first 300 chars):')
print('-' * 40)
print(cleaned[:300])

## Step 3 — Chunk the Document

`RecursiveChunker` splits text on progressively finer natural boundaries:

1. Paragraph breaks (`\n\n`)
2. Line breaks (`\n`)
3. Sentence endings (`. `, `! `, `? `)
4. Word spaces (` `)
5. Characters (last resort)

Small pieces are **merged** into chunks of approximately `chunk_size` characters,
with the tail of each chunk repeated at the start of the next for **overlap continuity**.

Each `Chunk` carries: `chunk_id`, `doc_id`, `text`, and optional `metadata`.

In [ ]:
from src.chunking.recursive_chunker import RecursiveChunker

chunker = RecursiveChunker(chunk_size=300, chunk_overlap=50)
chunks = chunker.chunk(document)

print(f'chunk_size    : {chunker.chunk_size}')
print(f'chunk_overlap : {chunker.chunk_overlap}')
print(f'Chunks created: {len(chunks)}')
print()

for i, chunk in enumerate(chunks):
    print(f'[Chunk {i}]  id={chunk.chunk_id}  doc_id={chunk.doc_id}  len={len(chunk.text)}')
    print(f'  {chunk.text[:90]}...' if len(chunk.text) > 90 else f'  {chunk.text}')
    print()

## Step 4 — Generate Mock Embeddings

`MockEmbeddingProvider` embeds a list of texts and returns one **384-dimensional
L2-normalised unit vector** per text.

**How it works:**
1. Compute SHA-256 digest of the text
2. Use the first 4 bytes as a seed for `numpy.random.default_rng`
3. Draw a 384-element standard-normal vector
4. L2-normalise to unit length

The same text always produces the same vector — the process is fully deterministic.
After embedding, each `Chunk` has its `embedding` field set.

In [ ]:
from src.embeddings.mock_embedding_provider import MockEmbeddingProvider

provider = MockEmbeddingProvider()
print(f'Provider  : {provider.provider_name}')
print(f'Dimension : {provider.dimension}')

# Embed all chunk texts in one batch call
texts = [c.text for c in chunks]
embeddings = provider.embed(texts)

# Attach embeddings back to each Chunk
for chunk, embedding in zip(chunks, embeddings):
    chunk.embedding = embedding

# Inspect the first embedding
import math
vec = embeddings[0]
norm = math.sqrt(sum(x * x for x in vec))
print(f'\nFirst embedding:')
print(f'  dim      : {len(vec)}')
print(f'  L2 norm  : {norm:.6f}  (should be ~1.0)')
print(f'  values[0:6]: {[round(v, 4) for v in vec[:6]]}')

# Verify determinism: same text -> same vector
vec2 = provider.embed([chunks[0].text])[0]
assert vec == vec2, 'Determinism check failed!'
print('\nDeterminism check: PASS (same text -> identical vector)')

## Step 5 — Store Chunks in ChromaDB

`ChromaVectorStore` wraps ChromaDB's `PersistentClient` with cosine similarity space.

Key behaviours:
- `add()` uses **upsert** — re-indexing the same `chunk_id` is idempotent
- Metadata fields are flattened to primitives (str / int / float / bool) before storage
- `doc_id` is injected into the metadata dict and popped back out on retrieval
- Score returned by `search()` is `1.0 - cosine_distance` (range: −1.0 to 1.0)

If `chromadb` is not installed, this cell reports the limitation and continues.

In [ ]:
try:
    import chromadb
    _CHROMA_AVAILABLE = True
except ImportError:
    _CHROMA_AVAILABLE = False
    print('chromadb is not installed.')
    print('Install it with:  pip install chromadb')
    print('Continuing — Step 6 will use a fresh pipeline that handles the store internally.')

if _CHROMA_AVAILABLE:
    from src.vectorstores.chroma_store import ChromaVectorStore

    _chroma_dir = os.path.join(_tmpdir, 'chroma_step5')
    store = ChromaVectorStore(
        persist_directory=_chroma_dir,
        collection_name='demo_step5',
    )

    store.add(chunks)
    print(f'Store     : {store.store_name}')
    print(f'Persist   : {_chroma_dir}')
    print(f'Chunks    : {len(chunks)} added')

    # Spot-check: search for the first chunk's own embedding
    results = store.search(chunks[0].embedding, top_k=3)
    print(f'\nSpot-check: search with chunk[0] embedding (expect rank-1 hit):')
    for r in results:
        print(f'  rank={r.rank}  score={r.score:.4f}  chunk_id={r.chunk_id}')

## Step 6 — Run a Retrieval Query via RAGPipeline

`RAGPipeline` is the orchestrator that performs all five steps above in one call to
`build_index()`. It then exposes `query(text, top_k)` for retrieval.

Using the pipeline directly (rather than individual components) is the recommended
pattern for both the Streamlit UI and evaluation runs — it guarantees the same
`EmbeddingProvider` instance is used for both indexing and querying.

In [ ]:
from src.pipeline.rag_pipeline import RAGPipeline
from src.embeddings.mock_embedding_provider import MockEmbeddingProvider
from src.chunking.recursive_chunker import RecursiveChunker
from src.observability.metrics import METRICS

# Reset metrics from the earlier steps
METRICS.reset()

if _CHROMA_AVAILABLE:
    from src.vectorstores.chroma_store import ChromaVectorStore

    _pipeline_chroma_dir = os.path.join(_tmpdir, 'chroma_pipeline')
    pipeline = RAGPipeline(
        embedding_provider=MockEmbeddingProvider(),
        vector_store=ChromaVectorStore(
            persist_directory=_pipeline_chroma_dir,
            collection_name='demo_pipeline',
        ),
        chunker=RecursiveChunker(chunk_size=300, chunk_overlap=50),
    )
else:
    # Fallback: FAISS in-memory (no persistence needed for this demo)
    try:
        from src.vectorstores.faiss_store import FaissVectorStore
        pipeline = RAGPipeline(
            embedding_provider=MockEmbeddingProvider(),
            vector_store=FaissVectorStore(),
            chunker=RecursiveChunker(chunk_size=300, chunk_overlap=50),
        )
    except ImportError:
        raise RuntimeError(
            'Neither chromadb nor faiss-cpu is installed.\n'
            'Install one with:  pip install chromadb   or   pip install faiss-cpu'
        )

# build_index() runs: load -> clean -> chunk -> embed -> store
index_result = pipeline.build_index(_doc_path)

print('Index build result:')
for key, val in index_result.items():
    print(f'  {key:<22}: {val}')

In [ ]:
QUERY = 'What chunking strategies are supported and how do they differ?'
TOP_K = 3

retrieved = pipeline.query(QUERY, top_k=TOP_K)

print(f'Query : {QUERY}')
print(f'top_k : {TOP_K}')
print(f'Retrieved {len(retrieved)} chunk(s)')

## Step 7 — Inspect Retrieved Chunks

Each returned `Chunk` has four retrieval fields populated:

| Field | Description |
|-------|-------------|
| `rank` | Position in the result list (1 = most similar) |
| `score` | Cosine similarity between query and chunk embeddings |
| `doc_id` | Source document identifier (derived from filename) |
| `chunk_id` | Unique chunk identifier within the document |

In [ ]:
print(f'Query: "{QUERY}"')
print('=' * 60)

for chunk in retrieved:
    print(f'Rank     : {chunk.rank}')
    print(f'Score    : {chunk.score:.4f}')
    print(f'doc_id   : {chunk.doc_id}')
    print(f'chunk_id : {chunk.chunk_id}')
    print(f'Text preview:')
    preview = chunk.text[:200].replace('\n', ' ')
    print(f'  {preview}...' if len(chunk.text) > 200 else f'  {preview}')
    print('-' * 60)

## Summary

This notebook demonstrated the complete RAG retrieval pipeline in mock mode:

1. **Document Loader** — reads a file and returns a `Document` with a derived `doc_id`
2. **Text Cleaner** — normalises whitespace; optional PII redaction
3. **RecursiveChunker** — splits on natural boundaries into overlapping chunks
4. **MockEmbeddingProvider** — deterministic 384-dim unit vectors, no API keys needed
5. **ChromaVectorStore** — cosine-similarity search with idempotent upsert
6. **RAGPipeline** — thin orchestrator that wires all five layers
7. **Retriever** — embeds query, searches store, populates rank and score

### Next Steps

- **Evaluate retrieval quality**: see `notebooks/rag_evaluation_experiments.ipynb`
- **Switch embedding provider**: replace `MockEmbeddingProvider()` with
  `HuggingFaceEmbeddingProvider()` or `OpenAIEmbeddingProvider()`
- **Compare chunking strategies**: swap `RecursiveChunker` for `FixedChunker`
  and re-run evaluation to compare metric scores
- **Try the Streamlit UI**: run `streamlit run app/streamlit_app.py` from the repo root

In [ ]:
# ---------------------------------------------------------------------------
# Cleanup and METRICS snapshot
# ---------------------------------------------------------------------------
import shutil

print('METRICS snapshot after this notebook run:')
print('-' * 40)
for key, val in METRICS.to_dict().items():
    print(f'  {key:<35}: {val}')

# Clean up temporary files
METRICS.reset()
shutil.rmtree(_tmpdir, ignore_errors=True)

print('\nTemporary files removed. METRICS reset.')